# 15 · Spark MLlib — Machine Learning Distribuído com `pyspark.ml`

🎯 **Objetivo:** sair da teoria do módulo de Spark MLlib e treinar, avaliar e colocar em
produção três modelos de verdade — **sem sair do universo de dados que você já conhece**
desde o notebook 00: as mesmas tabelas `empresas`, `funcionarios`, `vendas` e `avaliacoes`
usadas nos 14 notebooks anteriores. Nenhum dataset novo, nenhuma dependência de internet —
só a pergunta muda: em vez de "some, filtre, junte", agora é "aprenda um padrão e generalize
para dados novos".

**Teoria:** módulo *Spark MLlib* do curso (Transformer/Estimator/Pipeline, engenharia de
atributos, classificação, regressão, avaliação, tuning e aprendizado não supervisionado).

Rodamos inteiramente em `local[*]`, sem Docker — mesma infraestrutura dos notebooks 12-14.
São **três problemas de negócio**, cada um explorando uma face diferente da MLlib:

1. **Classificação** — qual avaliação de cliente vai ser negativa? (risco em tempo real)
2. **Regressão** — auditoria de equidade salarial (o cargo explica o salário — o resto é ruído?)
3. **Não supervisionado** — segmentar vendedores por perfil, sem olhar o cargo, e conferir se a máquina "redescobre" a hierarquia sozinha

Um bônus de recomendação (ALS) e um "Museu dos Erros" fecham o notebook. O notebook 16
retoma o modelo de classificação treinado aqui e o aplica **dentro de um stream** — o mesmo
padrão de simulação de arquivos dos notebooks 12-14.

---
### 🔤 O que você vai praticar

1. **Anatomia da API `pyspark.ml`** — Transformer, Estimator, Pipeline e Evaluator, na prática
2. **Engenharia de atributos** — `StringIndexer`, `OneHotEncoder`, `Imputer` (com indicador de nulo) e `VectorAssembler`, além de featurização de texto (`Tokenizer`, `StopWordsRemover`, `CountVectorizer`, `IDF`)
3. **Classificação** — `LogisticRegression`, `RandomForestClassifier` e `GBTClassifier`, com `CrossValidator` e `ParamGridBuilder`
4. **Avaliação sob desbalanceamento** — `BinaryClassificationEvaluator`, `MulticlassClassificationEvaluator`, matriz de confusão e limiar de decisão orientado a negócio
5. **Regressão regularizada** — `LinearRegression` com L1/L2/ElasticNet vs. `RandomForestRegressor`, e por que o L1 zera coeficientes de atributos irrelevantes
6. **Não supervisionado** — `KMeans`, método do cotovelo, `ClusteringEvaluator` (silhueta) e `PCA` para visualizar em 2D
7. **Persistência** — salvar e recarregar um `PipelineModel` treinado, pronto para o notebook 16

Vamos ao laboratório!

## Setup: SparkSession local e as 4 tabelas do curso

Nada de novo aqui — é a mesma sessão local dos notebooks 12-14. Se `make generate-data`
ainda não rodou neste ambiente, volte ao notebook 00 antes de continuar.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("app-01")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

spark

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import pyspark.sql.functions as F
from pyspark.ml import Pipeline

empresas = spark.read.parquet("../data/bronze/empresas")
funcionarios = spark.read.parquet("../data/bronze/funcionarios")
vendas = spark.read.parquet("../data/bronze/vendas")
avaliacoes = spark.read.parquet("../data/silver/avaliacoes")  # unificada no notebook 09

for nome, sdf in [("empresas", empresas), ("funcionarios", funcionarios),
                  ("vendas", vendas), ("avaliacoes", avaliacoes)]:
    print(f"{nome:14s} {sdf.count():>8,} linhas")

## Anatomia da API: Transformer, Estimator, Pipeline e Evaluator

Antes de treinar qualquer coisa em escala, os quatro conceitos que explicam a MLlib
inteira — em quatro linhas de código, sobre um DataFrame de brinquedo.

In [ ]:
from pyspark.ml.feature import Tokenizer

juguete = spark.createDataFrame(
    [(1, "Ótimo atendimento, super recomendo"), (2, "Péssimo, muito demorado")],
    ["id", "comentario"],
)

# Transformer: a lógica é fixa ("quebre a string nos espaços") — não há nada a
# aprender com os dados, então não existe .fit(), só .transform()
tokenizer = Tokenizer(inputCol="comentario", outputCol="palavras")
tokenizer.transform(juguete).show(truncate=False)

In [ ]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.linalg import Vectors

juguete_num = spark.createDataFrame(
    [(1, Vectors.dense([10.0, 200.0])), (2, Vectors.dense([12.0, 190.0])),
     (3, Vectors.dense([9.0, 250.0]))],
    ["id", "features"],
)

# Estimator: precisa .fit() para APRENDER média/desvio-padrão dos dados...
escalador = StandardScaler(inputCol="features", outputCol="features_esc")
escalador_treinado = escalador.fit(juguete_num)  # -> StandardScalerModel

# ...o resultado do .fit() é um MODEL, que por sua vez é um Transformer
print(type(escalador).__name__, "->", type(escalador_treinado).__name__)
escalador_treinado.transform(juguete_num).show(truncate=False)

📌 **A convenção de nomes é literal:** se a classe termina em `Model`, ela é um
Transformer já treinado. Se não termina, é um Estimator que ainda precisa de `.fit()`.
Um **Pipeline** encadeia os dois tipos como uma única unidade — e um **Evaluator**
fecha o ciclo, transformando previsões em um número. É só isso, em todo o notebook.

## O contrato dos dados: `features` e `label`

Todo algoritmo supervisionado da MLlib espera exatamente duas colunas: `features`
(um `Vector` de dimensão fixa) e `label` (`Double`). Os vetores existem em duas formas
matematicamente equivalentes — só muda o custo de armazenamento:

In [ ]:
denso = Vectors.dense([0.0, 0.0, 3.5, 0.0, 7.1])
esparso = Vectors.sparse(5, [2, 4], [3.5, 7.1])

print("Denso: ", denso)
print("Esparso:", esparso)
print("São iguais?", denso == esparso)

---
## Problema 1 · Classificação: risco de avaliação negativa

**O problema de negócio:** a cada avaliação que chega — pelo app, pelo site ou pelo call
center — queremos sinalizar, no instante em que ela chega, o **risco de ser uma nota ruim**
(`nota` ≤ 2), para que um atendente humano possa entrar em contato de recuperação antes que
o cliente insatisfeito já tenha decidido cancelar. É a mesma tabela `avaliacoes` que o
notebook 09 leu de 3 formatos diferentes — inclusive o mesmo bug de versão do app (`4.9.1`)
que aquele notebook só *descreveu*. Aqui, o modelo vai *descobrir* esse padrão sozinho.

### Engenharia de atributos

Três decisões de featurização, na ordem em que o módulo teórico recomenda —
**indexar → codificar → imputar → montar → escalar**:

- `tempo_atendimento_min` só existe para avaliações do Call Center — as demais vêm `null`.
  Antes de imputar, criamos um indicador de ausência: o próprio fato de o dado faltar
  pode carregar informação (é *literalmente* o canal, aqui).
- `versao_app` só existe para o canal App — preenchemos com um rótulo explícito
  (`"nao_app"`) em vez de deixar nulo, porque o `StringIndexer` não sabe lidar com `null`.
- O rótulo (`label`) é derivado, não present no dado bruto: `nota <= 2`.

In [ ]:
avaliacoes_feat = (
    avaliacoes
    .withColumn("label", (F.col("nota") <= 2).cast("double"))
    .withColumn("tempo_atendimento_ausente", F.col("tempo_atendimento_min").isNull().cast("double"))
    .fillna({"versao_app": "nao_app"})
)

taxa_positivos = avaliacoes_feat.agg(F.avg("label")).first()[0]
print(f"Taxa de 'nota ruim' na base inteira: {taxa_positivos:.1%}  (classes desbalanceadas)")

treino, teste = avaliacoes_feat.randomSplit([0.8, 0.2], seed=42)
print(f"Treino: {treino.count():,}  |  Teste: {teste.count():,}")

### Um desvio rápido: featurizando o `comentario` (e por que ele NÃO vai para o classificador)

O texto livre é uma fonte de sinal riquíssima em avaliações reais — vale conhecer a cadeia
clássica de NLP da MLlib mesmo assim.

In [ ]:
from pyspark.ml.feature import IDF, CountVectorizer, RegexTokenizer, StopWordsRemover

tokenizador = RegexTokenizer(inputCol="comentario", outputCol="tokens", pattern=r"\W+", toLowercase=True)
removedor = StopWordsRemover(
    inputCol="tokens", outputCol="tokens_limpos",
    stopWords=StopWordsRemover.loadDefaultStopWords("portuguese"),
)
contador_tf = CountVectorizer(inputCol="tokens_limpos", outputCol="tf", vocabSize=200, minDF=5.0)
idf = IDF(inputCol="tf", outputCol="features_texto", minDocFreq=5)

pipeline_texto = Pipeline(stages=[tokenizador, removedor, contador_tf, idf])
modelo_texto = pipeline_texto.fit(avaliacoes_feat)
texto_transformado = modelo_texto.transform(avaliacoes_feat)

vocabulario = modelo_texto.stages[2].vocabulary
print(f"Vocabulário aprendido ({len(vocabulario)} termos): {vocabulario[:12]}")
texto_transformado.select("nota", "comentario", "features_texto").show(5, truncate=60)

📌 **Por que este `features_texto` fica de fora do classificador principal:** nosso
`comentario` sintético foi gerado a partir de um punhado fixo de frases por nota (veja
`scripts/generate_dataset.py`) — colar o TF-IDF direto no classificador tornaria o problema
artificialmente perfeito, um viés que não existiria com comentários reais e livres. Para
manter o problema no nível de dificuldade de um cenário de produção, o classificador abaixo
usa só `canal`, `versao_app` e `tempo_atendimento_min` — sinais estruturados, genuinamente
imperfeitos.

### A armadilha do vazamento de dados (*data leakage*)

Antes de montar o pipeline de verdade, vale registrar o erro conceitual mais comum:

```python
# ❌ ERRADO — o StringIndexer "vê" o vocabulário do conjunto de teste antes do split
modelo_idx = StringIndexer(...).fit(avaliacoes_feat)          # fit em TUDO
treino, teste = modelo_idx.transform(avaliacoes_feat).randomSplit([0.8, 0.2])

# ✅ CORRETO — como fizemos acima: split primeiro, fit só no treino
treino, teste = avaliacoes_feat.randomSplit([0.8, 0.2], seed=42)
modelo_idx = StringIndexer(...).fit(treino)                    # fit só no treino
```

Isso vale para **todo** Estimator: `Imputer` (a mediana), `StringIndexer` (o vocabulário),
`StandardScaler` (média/desvio). É por isso que o `Pipeline` existe — ele torna o
vazamento estruturalmente difícil de cometer, desde que `.fit()` só seja chamado sobre o
treino.

In [ ]:
from pyspark.ml.feature import (
    Imputer,
    OneHotEncoder,
    StandardScaler,
    StringIndexer,
    VectorAssembler,
)

estagios_features = [
    StringIndexer(inputCols=["canal", "versao_app"], outputCols=["canal_idx", "versao_app_idx"],
                  handleInvalid="keep"),
    OneHotEncoder(inputCols=["canal_idx", "versao_app_idx"], outputCols=["canal_ohe", "versao_app_ohe"]),
    Imputer(inputCols=["tempo_atendimento_min"], outputCols=["tempo_atendimento_imp"], strategy="median"),
    VectorAssembler(
        inputCols=["canal_ohe", "versao_app_ohe", "tempo_atendimento_imp", "tempo_atendimento_ausente"],
        outputCol="features_bruto", handleInvalid="skip",
    ),
    # withMean=False é proposital: canal_ohe/versao_app_ohe são esparsos, e subtrair a
    # média densificaria o vetor inteiro (a armadilha clássica descrita no módulo teórico)
    StandardScaler(inputCol="features_bruto", outputCol="features", withStd=True, withMean=False),
]

pipeline_features = Pipeline(stages=estagios_features)
modelo_features = pipeline_features.fit(treino)  # só enxerga o treino

treino_feat = modelo_features.transform(treino).cache()
teste_feat = modelo_features.transform(teste).cache()
treino_feat.count(), teste_feat.count()

### Comparando três famílias de classificadores

Com os atributos já prontos (`treino_feat`/`teste_feat`), a iteração de modelos fica barata:
não recomputamos indexação, one-hot ou imputação a cada tentativa — só o `.fit()` do
classificador em si.

In [ ]:
from pyspark.ml.classification import GBTClassifier, LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

avaliador_roc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
avaliador_pr = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderPR")

candidatos = {
    "Regressão Logística": LogisticRegression(featuresCol="features", labelCol="label", maxIter=50),
    "Random Forest": RandomForestClassifier(featuresCol="features", labelCol="label",
                                             numTrees=100, maxDepth=6, seed=42),
    "GBT": GBTClassifier(featuresCol="features", labelCol="label", maxIter=50, maxDepth=4, seed=42),
}

resultados = []
for nome, classificador in candidatos.items():
    modelo = classificador.fit(treino_feat)
    previsoes = modelo.transform(teste_feat)
    resultados.append({
        "modelo": nome,
        "AUC-ROC": avaliador_roc.evaluate(previsoes),
        "AUC-PR": avaliador_pr.evaluate(previsoes),
    })

pd.DataFrame(resultados).sort_values("AUC-PR", ascending=False).reset_index(drop=True)

📌 Modelos de árvore (Random Forest, GBT) tendem a se sair bem aqui porque lidam
naturalmente com o padrão "tudo-ou-nada" do bug da versão `4.9.1` — uma regressão logística
também aprende, mas via um coeficiente enorme numa única categoria. Seguimos com
**Random Forest**: competitivo e diretamente interpretável via `featureImportances`.

### Ajuste de hiperparâmetros com `CrossValidator`

Desta vez sobre o **Pipeline inteiro** (features + classificador) — a validação cruzada
refaz `.fit()` de *todos* os estágios em cada fold, o único jeito de garantir que a busca
de hiperparâmetros também não vaze dados do teste.

In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

rf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=42)
pipeline_completo = Pipeline(stages=estagios_features + [rf])

grade = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 150])
    .addGrid(rf.maxDepth, [4, 8])
    .build()
)

validador_cruzado = CrossValidator(
    estimator=pipeline_completo,
    estimatorParamMaps=grade,
    evaluator=avaliador_pr,
    numFolds=3,
    parallelism=2,
    seed=42,
)

cv_modelo = validador_cruzado.fit(treino)  # sobre o DataFrame bruto — o Pipeline cuida do resto
melhor_pipeline = cv_modelo.bestModel
print("Melhor combinação encontrada — numTrees/maxDepth do estágio de classificação:")
print(melhor_pipeline.stages[-1].extractParamMap())

### Avaliação final no conjunto de teste

In [ ]:
previsoes_finais = melhor_pipeline.transform(teste)

print(f"AUC-ROC: {avaliador_roc.evaluate(previsoes_finais):.4f}")
print(f"AUC-PR : {avaliador_pr.evaluate(previsoes_finais):.4f}")

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

for metrica in ["weightedPrecision", "weightedRecall", "f1"]:
    valor = MulticlassClassificationEvaluator(labelCol="label", metricName=metrica).evaluate(previsoes_finais)
    print(f"{metrica:18s} {valor:.4f}")

recall_nota_ruim = MulticlassClassificationEvaluator(
    labelCol="label", metricName="recallByLabel", metricLabel=1.0
).evaluate(previsoes_finais)
print(f"{'recall (nota ruim)':18s} {recall_nota_ruim:.4f}")

print("\nMatriz de confusão (label real x previsão):")
previsoes_finais.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

📌 **Por que olhar `recallByLabel` e não só a acurácia:** com ~72% das avaliações sendo
"nota boa", um classificador preguiçoso que sempre prevê 0 já acertaria ~72% — e teria
recall **zero** para a classe que realmente importa. Voltamos a isso no Museu dos Erros.

### O limiar é uma decisão de negócio, não de estatística

`probability[1]` é a chance estimada de a avaliação ser ruim. Transformar isso em "ligar
ou não ligar" para o cliente é uma escolha de **custo esperado**, não de acurácia:

In [ ]:
from pyspark.ml.functions import vector_to_array

CUSTO_CONTATO = 5.0             # custo de uma ligação de retenção proativa (hipotético, didático)
CUSTO_CLIENTE_NAO_CONTATADO = 300.0  # valor esperado perdido se uma nota ruim não gerar contato

com_prob = previsoes_finais.withColumn("prob_risco", vector_to_array("probability")[1])

custos_por_limiar = []
for limiar in [i / 20 for i in range(1, 20)]:
    sinalizados = com_prob.withColumn("contato", (F.col("prob_risco") >= limiar).cast("int"))
    agregado = sinalizados.agg(
        F.sum("contato").alias("n_contatos"),
        F.sum(F.when((F.col("contato") == 0) & (F.col("label") == 1.0), 1).otherwise(0)).alias("n_perdidos"),
    ).first()
    custo_total = agregado["n_contatos"] * CUSTO_CONTATO + agregado["n_perdidos"] * CUSTO_CLIENTE_NAO_CONTATADO
    custos_por_limiar.append((limiar, custo_total))

melhor_limiar, menor_custo = min(custos_por_limiar, key=lambda par: par[1])
print(f"Limiar de negócio ótimo: {melhor_limiar:.2f}  |  Custo esperado: R$ {menor_custo:,.2f}")
print(f"(o limiar 'estatístico' padrão de 0.5 custaria R$ "
      f"{dict(custos_por_limiar)[0.5]:,.2f})")

### A descoberta: o modelo re-encontra o bug do notebook 09 sozinho

O notebook 09 *descreveu* que a versão `4.9.1` do app tinha um bug que derrubava as notas.
Vamos conferir se o classificador aprendeu isso — sem que a gente tenha dito a ele.

In [ ]:
rf_treinado = melhor_pipeline.stages[-1]
print("Vetor de importância de atributos (por índice, sem nomes):")
print(rf_treinado.featureImportances)

In [ ]:
print("Risco médio PREVISTO vs. taxa REAL de nota ruim, por versão do app:")
com_prob.groupBy("versao_app").agg(
    F.round(F.avg("prob_risco"), 3).alias("risco_previsto_medio"),
    F.round(F.avg("label"), 3).alias("taxa_real_nota_ruim"),
    F.count("*").alias("n"),
).orderBy("versao_app").show()

📌 **O modelo redescobriu, sozinho, exatamente o achado do notebook 09:** avaliações da
versão `4.9.1` recebem um risco previsto muito acima das demais versões — e bate com a
taxa real. Nenhuma regra foi escrita à mão; o `RandomForestClassifier` aprendeu o padrão
só de olhar `canal` + `versao_app` + `tempo_atendimento_min`.

### Persistindo o Pipeline treinado

O artefato salvo carrega **todos** os estágios — vocabulário do `StringIndexer`, mediana
do `Imputer`, desvio-padrão do `StandardScaler` e as árvores do `RandomForestClassifier`.
O notebook 16 vai recarregá-lo para aplicar em um stream.

In [ ]:
CAMINHO_MODELO_RISCO = "../data/models/nb15_risco_avaliacao"

melhor_pipeline.write().overwrite().save(CAMINHO_MODELO_RISCO)
print(f"Pipeline salvo em {CAMINHO_MODELO_RISCO}")

---
## Problema 2 · Regressão: auditoria de equidade salarial

**O problema de negócio:** RH quer saber se a política salarial é justa — o salário de
cada funcionário deveria ser bem explicado pelo **cargo** (e talvez pela empresa/região),
não por fatores arbitrários. Vamos treinar um modelo que prevê o salário *esperado* dado
o perfil do funcionário e usar o **resíduo** (salário real − salário previsto) como um
detector de anomalias: quem está pagando muito acima ou abaixo do esperado para o cargo?

In [ ]:
REFERENCIA = F.to_date(F.lit("2026-07-01"))  # mesmo corte usado por scripts/generate_dataset.py
# (evita current_date(): a data de geração é fixa, então a data de referência da
# featurização também precisa ser fixa, ou tempo_casa_meses mudaria a cada execução)

funcionarios_sal = (
    funcionarios
    .join(empresas, "id_empresa")
    .withColumn("tempo_casa_meses", F.months_between(REFERENCIA, F.col("data_admissao")))
    .withColumnRenamed("salario", "label")
)
funcionarios_sal.select("cargo", "label", "tempo_casa_meses", "setor", "regiao").show(5)

treino_sal, teste_sal = funcionarios_sal.randomSplit([0.8, 0.2], seed=42)

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import LinearRegression, RandomForestRegressor

estagios_sal = [
    StringIndexer(inputCols=["cargo", "setor", "regiao"],
                  outputCols=["cargo_idx", "setor_idx", "regiao_idx"], handleInvalid="keep"),
    OneHotEncoder(inputCols=["cargo_idx", "setor_idx", "regiao_idx"],
                  outputCols=["cargo_ohe", "setor_ohe", "regiao_ohe"]),
    VectorAssembler(inputCols=["cargo_ohe", "setor_ohe", "regiao_ohe", "tempo_casa_meses"],
                     outputCol="features_bruto", handleInvalid="skip"),
    StandardScaler(inputCol="features_bruto", outputCol="features", withStd=True, withMean=False),
]

avaliador_rmse = RegressionEvaluator(labelCol="label", metricName="rmse")
avaliador_r2 = RegressionEvaluator(labelCol="label", metricName="r2")

variantes_sal = {
    "Ridge (L2)": LinearRegression(featuresCol="features", labelCol="label", regParam=0.3, elasticNetParam=0.0),
    "Lasso (L1)": LinearRegression(featuresCol="features", labelCol="label", regParam=0.3, elasticNetParam=1.0),
    "ElasticNet": LinearRegression(featuresCol="features", labelCol="label", regParam=0.3, elasticNetParam=0.5),
    "Random Forest": RandomForestRegressor(featuresCol="features", labelCol="label", numTrees=100, seed=42),
}

resultados_sal, pipelines_sal = [], {}
for nome, modelo_base in variantes_sal.items():
    ajustado = Pipeline(stages=estagios_sal + [modelo_base]).fit(treino_sal)
    previsoes = ajustado.transform(teste_sal)
    resultados_sal.append({"modelo": nome,
                            "RMSE": avaliador_rmse.evaluate(previsoes),
                            "R2": avaliador_r2.evaluate(previsoes)})
    pipelines_sal[nome] = ajustado

pd.DataFrame(resultados_sal)

### O que o L1 zera — e o que isso significa

Se `tempo_casa_meses`, `setor` e `regiao` realmente não influenciam o salário (por
construção dos dados, não deveriam), o Lasso (L1) deveria zerar boa parte desses
coeficientes, enquanto o Ridge (L2) apenas os encolhe sem zerar.

In [ ]:
lasso = pipelines_sal["Lasso (L1)"].stages[-1]
ridge = pipelines_sal["Ridge (L2)"].stages[-1]

n_zero_lasso = int((lasso.coefficients.toArray() == 0).sum())
n_zero_ridge = int((ridge.coefficients.toArray() == 0).sum())
total = len(lasso.coefficients)

print(f"Lasso (L1): {n_zero_lasso}/{total} coeficientes exatamente zero")
print(f"Ridge (L2): {n_zero_ridge}/{total} coeficientes exatamente zero")

📌 **A confirmação estatística de uma decisão de política de RH:** o L1 zera a maior
parte dos coeficientes de `setor`/`regiao`/`tempo_casa_meses` — a regularização "descobre",
sem que ninguém tenha dito, que **o salário aqui é determinado pelo cargo**, não pelo
tempo de casa. Se isso não bater com a política declarada da empresa, é um achado de
auditoria por si só.

### Auditoria: os maiores desvios entre salário real e esperado

In [ ]:
previsoes_rf_sal = pipelines_sal["Random Forest"].transform(teste_sal)

auditoria = (
    previsoes_rf_sal
    .withColumn("residual", F.col("label") - F.col("prediction"))
    .select("nome_funcionario", "cargo", "setor", "regiao", "label", "prediction", "residual")
    .orderBy(F.abs(F.col("residual")).desc())
)
auditoria.show(10, truncate=False)

In [ ]:
amostra_sal = previsoes_rf_sal.select("label", "prediction", "cargo").toPandas()

fig, ax = plt.subplots(figsize=(7, 6))
for cargo, grupo in amostra_sal.groupby("cargo"):
    ax.scatter(grupo["label"], grupo["prediction"], label=cargo, alpha=0.4, s=12)
limites = [amostra_sal["label"].min(), amostra_sal["label"].max()]
ax.plot(limites, limites, "k--", linewidth=1, label="previsão perfeita")
ax.set_xlabel("Salário real (R$)")
ax.set_ylabel("Salário previsto (R$)")
ax.set_title("Auditoria de equidade salarial: previsto vs. real")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

---
## Problema 3 · Não supervisionado: segmentando vendedores por perfil

**O problema de negócio:** montar trilhas de incentivo/treinamento diferentes por perfil
de vendedor. Em vez de simplesmente usar o `cargo` (que já conhecemos), o teste real do
clustering é: **se escondermos o cargo do modelo**, ele redescobre uma segmentação
parecida só olhando para salário, tempo de casa e comportamento de vendas?

In [ ]:
vendas_por_funcionario = vendas.groupBy("id_funcionario").agg(
    F.count("*").alias("n_vendas"),
    F.sum("valor").alias("valor_total"),
    F.avg("valor").alias("ticket_medio"),
)

vendedores = (
    funcionarios
    .join(vendas_por_funcionario, "id_funcionario")
    .withColumn("tempo_casa_meses", F.months_between(REFERENCIA, F.col("data_admissao")))
    .select("id_funcionario", "nome_funcionario", "cargo", "salario", "tempo_casa_meses",
            "n_vendas", "valor_total", "ticket_medio")
)
print(f"{vendedores.count():,} vendedores com histórico de vendas")
vendedores.show(5)

In [ ]:
montador_seg = VectorAssembler(
    inputCols=["salario", "tempo_casa_meses", "n_vendas", "valor_total", "ticket_medio"],
    outputCol="features_bruto",
)
# withMean=True aqui é seguro: não há one-hot nem vetores esparsos nesta featurização,
# só atributos numéricos densos — e o K-Means, baseado em distância euclidiana, PRECISA
# de escala comparável entre salário (milhares) e n_vendas (dezenas)
escalador_seg = StandardScaler(inputCol="features_bruto", outputCol="features", withStd=True, withMean=True)

vendedores_esc = Pipeline(stages=[montador_seg, escalador_seg]).fit(vendedores).transform(vendedores).cache()
vendedores_esc.count()

### Escolhendo `k`: cotovelo e silhueta

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

avaliador_cluster = ClusteringEvaluator(featuresCol="features", predictionCol="cluster", metricName="silhouette")

custos, silhuetas = [], []
for k in range(2, 9):
    modelo_k = KMeans(featuresCol="features", predictionCol="cluster", k=k, seed=42).fit(vendedores_esc)
    previsto_k = modelo_k.transform(vendedores_esc)
    custos.append((k, modelo_k.summary.trainingCost))
    silhuetas.append((k, avaliador_cluster.evaluate(previsto_k)))

df_custos = pd.DataFrame(custos, columns=["k", "custo"])
df_silhuetas = pd.DataFrame(silhuetas, columns=["k", "silhueta"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(df_custos["k"], df_custos["custo"], marker="o")
ax1.set_title("Método do cotovelo (WSSSE)")
ax1.set_xlabel("k")
ax1.set_ylabel("Custo intra-cluster")
ax2.plot(df_silhuetas["k"], df_silhuetas["silhueta"], marker="o", color="darkorange")
ax2.set_title("Coeficiente de silhueta")
ax2.set_xlabel("k")
ax2.set_ylabel("Silhueta média")
plt.tight_layout()
plt.show()

In [ ]:
melhor_k = max(silhuetas, key=lambda par: par[1])[0]
print(f"k escolhido pela silhueta: {melhor_k}")

modelo_final_k = KMeans(featuresCol="features", predictionCol="cluster", k=melhor_k, seed=42).fit(vendedores_esc)
segmentados = modelo_final_k.transform(vendedores_esc)

In [ ]:
from pyspark.ml.feature import PCA

modelo_pca = PCA(k=2, inputCol="features", outputCol="pca_features").fit(segmentados)
projetado = modelo_pca.transform(segmentados).select("pca_features", "cluster", "cargo").toPandas()
projetado[["pc1", "pc2"]] = pd.DataFrame(projetado["pca_features"].apply(lambda v: v.toArray()).tolist())

fig, ax = plt.subplots(figsize=(7, 6))
disp = ax.scatter(projetado["pc1"], projetado["pc2"], c=projetado["cluster"], cmap="tab10", alpha=0.5, s=12)
ax.set_xlabel("Componente Principal 1")
ax.set_ylabel("Componente Principal 2")
ax.set_title(f"Segmentação de vendedores (k={melhor_k}) projetada em 2D via PCA")
plt.colorbar(disp, label="cluster")
plt.tight_layout()
plt.show()

### Interpretando os segmentos: a máquina redescobriu o cargo? (spoiler: não)

In [ ]:
print("Composição de cada cluster por cargo (contagem):")
segmentados.groupBy("cluster").pivot("cargo").count().orderBy("cluster").show(truncate=False)

print("Perfil médio de cada cluster:")
segmentados.groupBy("cluster").agg(
    F.count("*").alias("n"),
    F.round(F.avg("salario"), 2).alias("salario_medio"),
    F.round(F.avg("tempo_casa_meses"), 1).alias("tempo_casa_medio_meses"),
    F.round(F.avg("n_vendas"), 1).alias("n_vendas_media"),
).orderBy("cluster").show()

📌 **Resultado honesto: os dois clusters NÃO se alinham com o cargo** — cada cargo se
espalha quase 50/50 entre os dois grupos, e `salario_medio` é praticamente idêntico entre
eles. Isso **não é um bug do K-Means** — é a distância euclidiana fazendo exatamente o que
deveria com os atributos que demos a ela. O problema é a featurização, não o algoritmo.

### Diagnóstico: por que a segmentação falhou

O `StandardScaler` dá peso igual a cada atributo depois de padronizado. Se 4 dos 5
atributos forem **ruído** — sem relação real com o cargo — eles dominam a distância em
volume (4 contra 1) e afogam o único sinal genuíno (`salario`). Vamos conferir:

In [ ]:
print("Correlação de cada atributo comportamental com o salário:")
vendedores.select(
    F.corr("tempo_casa_meses", "salario").alias("tempo_casa_meses"),
    F.corr("n_vendas", "salario").alias("n_vendas"),
    F.corr("valor_total", "salario").alias("valor_total"),
    F.corr("ticket_medio", "salario").alias("ticket_medio"),
).show()

📌 Todas as correlações ficam perto de zero — **por construção dos dados** (o script
`generate_dataset.py` sorteia vendas de forma independente do funcionário). Em outras
palavras: pedimos ao K-Means para achar estrutura numa mistura de 20% sinal e 80% ruído,
e ele achou uma divisão qualquer nesse ruído. É a versão de clustering do "garbage in,
garbage out" — **seleção de atributos importa tanto no não supervisionado quanto no
supervisionado**.

### Segunda tentativa: isolando o único atributo com sinal real

Removendo o ruído e mantendo só `salario`. E desta vez vamos **ignorar a silhueta na
escolha de `k`** — ela tende a preferir o corte mais largo (Diretor Comercial vs. o
resto) em vez de uma segmentação de negócio útil. Como já sabemos, de fora, que existem
6 cargos, forçamos `k=6` deliberadamente e conferimos se a segmentação bate.

In [ ]:
montador_sal_only = VectorAssembler(inputCols=["salario"], outputCol="features_bruto")
escalador_sal_only = StandardScaler(inputCol="features_bruto", outputCol="features", withStd=True, withMean=True)

vendedores_sal_only = (
    Pipeline(stages=[montador_sal_only, escalador_sal_only]).fit(vendedores).transform(vendedores)
)
segmentados_v2 = (
    KMeans(featuresCol="features", predictionCol="cluster", k=6, seed=42)
    .fit(vendedores_sal_only)
    .transform(vendedores_sal_only)
)

contagens = segmentados_v2.groupBy("cluster", "cargo").count().toPandas()
pureza_por_cluster = contagens.groupby("cluster")["count"].max() / contagens.groupby("cluster")["count"].sum()
pureza_geral = contagens.groupby("cluster")["count"].max().sum() / contagens["count"].sum()

print(f"Pureza geral (fração de cada cluster que é o cargo majoritário): {pureza_geral:.1%}")
print("(um sorteio aleatório entre 6 cargos equilibrados acertaria só ~16.7%)")
segmentados_v2.groupBy("cluster").pivot("cargo").count().orderBy("cluster").show(truncate=False)

In [ ]:
amostra_sal_only = segmentados_v2.select("salario", "cargo", "cluster").toPandas()
ordem_cargos = (
    amostra_sal_only.groupby("cargo")["salario"].mean().sort_values().index.tolist()
)
amostra_sal_only["y_cargo"] = amostra_sal_only["cargo"].apply(ordem_cargos.index)

fig, ax = plt.subplots(figsize=(8, 5))
disp = ax.scatter(
    amostra_sal_only["salario"],
    amostra_sal_only["y_cargo"] + (amostra_sal_only["cluster"] - amostra_sal_only["cluster"].mean()) * 0.03,
    c=amostra_sal_only["cluster"], cmap="tab10", alpha=0.4, s=10,
)
ax.set_yticks(range(len(ordem_cargos)))
ax.set_yticklabels(ordem_cargos)
ax.set_xlabel("Salário (R$)")
ax.set_title("Cargo real (eixo Y) x cluster encontrado (cor) — onde o K-Means erra é onde as faixas se sobrepõem")
plt.colorbar(disp, label="cluster")
plt.tight_layout()
plt.show()

📌 **O ponto central do exercício:** removendo o ruído e usando só `salario`, o K-Means
recupera o cargo com uma pureza bem acima do acaso — mas **longe de perfeita**: as faixas
salariais de cargos vizinhos (ex.: Vendedor Junior/Pleno) se sobrepõem, e o algoritmo erra
justamente nessas bordas. As duas lições juntas valem mais que uma "vitória" limpa: **(1)**
atributos irrelevantes afogam sinal real sob distância euclidiana padronizada, e **(2)**
mesmo com o atributo certo, clustering raramente recupera um rótulo externo com pureza de
100% — e tudo bem, porque em produção normalmente não há rótulo nenhum para comparar.

---
## Bônus rápido: recomendação com ALS

Filtragem colaborativa não é o encaixe natural para esta base (o caso de uso raiz é
usuário × produto, com milhões de linhas) — mas dá para expor a API com o que temos:
"como uma empresa tende a avaliar cada canal de atendimento?" (`empresa` × `canal` × `nota`).
Trate isto como uma demonstração da API, não como uma recomendação de produção.

In [ ]:
from pyspark.ml.recommendation import ALS

indexador_canal = StringIndexer(inputCol="canal", outputCol="canal_idx")
avaliacoes_als = (
    indexador_canal.fit(avaliacoes)
    .transform(avaliacoes)
    .select(
        F.col("id_empresa").cast("int").alias("empresa_id"),
        F.col("canal_idx").cast("int"),
        F.col("nota").cast("float"),
    )
)
treino_als, teste_als = avaliacoes_als.randomSplit([0.8, 0.2], seed=42)

als = ALS(userCol="empresa_id", itemCol="canal_idx", ratingCol="nota",
          coldStartStrategy="drop", nonnegative=True, seed=42)
modelo_als = als.fit(treino_als)
previsoes_als = modelo_als.transform(teste_als)

rmse_als = RegressionEvaluator(labelCol="nota", predictionCol="prediction", metricName="rmse").evaluate(previsoes_als)
print(f"RMSE (nota prevista empresa x canal): {rmse_als:.3f}")

print("\nCanal com maior nota prevista, por empresa (top 5):")
modelo_als.recommendForAllUsers(1).show(5, truncate=False)

📌 `coldStartStrategy="drop"` descarta pares usuário/item que nunca apareceram no treino
(sem isso, o `.transform()` devolveria `NaN` para eles — um detalhe que derruba métricas
silenciosamente se ninguém filtrar antes de avaliar).

---
## 🏛️ O Museu dos Erros

Um resumo das armadilhas clássicas — todas com uma decisão correspondente neste mesmo
notebook.

In [ ]:
acuracia_trivial = 1 - taxa_positivos
print(f"Um classificador 'sempre nota boa' (nunca sinaliza risco) acerta {acuracia_trivial:.1%} das vezes...")
print("...só que tem recall 0% para exatamente a classe que motivou o projeto inteiro.")

- **A armadilha da acurácia sob desbalanceamento** — provada acima: 72% de acerto e zero
  utilidade de negócio. É por isso que avaliamos com `recallByLabel` e AUC-PR, não só acurácia.
- **Vazamento de dados** — `.fit()` de qualquer Estimator (`StringIndexer`, `Imputer`,
  `StandardScaler`...) só pode ver o conjunto de treino. Fizemos o split *antes* de montar
  o Pipeline, em ambos os problemas supervisionados.
- **`StandardScaler(withMean=True)` sobre vetores esparsos** — destrói a esparsidade
  (todo zero vira `-média`), inflando memória e rede. Por isso os Problemas 1 e 2 usam
  `withMean=False` (há one-hot no vetor) e só o Problema 3 usa `withMean=True` (vetor
  100% denso, sem one-hot).
- **`StringIndexer(handleInvalid="error")` em produção** — derruba o job inteiro diante de
  uma categoria nova. Usamos `"keep"` nos dois pipelines supervisionados.
- **Featurização "boa demais para ser verdade"** — o TF-IDF do `comentario` teria dado um
  classificador quase perfeito, porque o texto sintético é gerado a partir do próprio
  rótulo. Deixamos de fora por honestidade estatística — o achado da versão `4.9.1` já era
  convincente o bastante sem ele.

In [ ]:
spark.stop()

---
🎉 **Parabéns!** Você completou o notebook 15.

Você aprendeu:
- A anatomia da API `pyspark.ml`: **Transformer**, **Estimator**, **Pipeline** e **Evaluator**
- Engenharia de atributos completa: `StringIndexer`, `OneHotEncoder`, `Imputer` (com
  indicador de nulo), featurização de texto (`Tokenizer`/`StopWordsRemover`/`CountVectorizer`/`IDF`)
  e `VectorAssembler`
- Por que a ordem split → fit evita vazamento de dados, e por que `StandardScaler(withMean=True)`
  é perigoso sobre vetores esparsos
- Classificação com `LogisticRegression`, `RandomForestClassifier` e `GBTClassifier`,
  ajustada com `CrossValidator` + `ParamGridBuilder`
- Avaliação sob desbalanceamento: `BinaryClassificationEvaluator` (ROC/PR),
  `MulticlassClassificationEvaluator` por classe, matriz de confusão e limiar de negócio
- Regressão regularizada (`LinearRegression` L1/L2/ElasticNet) e por que o L1 zera
  coeficientes irrelevantes — uma auditoria de equidade salarial de verdade
- Clustering (`KMeans`), escolha de `k` (cotovelo + silhueta) e visualização — e por que
  atributos irrelevantes sob `StandardScaler` podem afogar o único sinal real
- Persistência de um `PipelineModel` treinado, pronto para o notebook 16

### 📝 Exercícios propostos

1. Troque `OneHotEncoder` por `FeatureHasher` no Problema 1 e compare o tempo de
   treinamento e o AUC-PR — em que ponto a colisão de hash começa a doer?
2. Substitua o `CrossValidator` por `TrainValidationSplit` no Problema 1 e compare o tempo
   total de ajuste — quantas vezes menos `.fit()` são executados?
3. No Problema 1, tente prever a **nota completa** (1 a 5) como um problema multiclasse
   em vez de binário — o que muda no `Evaluator` e na matriz de confusão?
4. No Problema 3, tente `PCA(k=1)` antes do `VectorAssembler` com todas as 5 features
   originais — o primeiro componente principal consegue isolar o sinal do salário em meio
   ao ruído melhor do que a distância euclidiana bruta conseguiu?
5. No Problema 1, treine também um `GBTClassifier` dentro do `CrossValidator` (em vez do
   Random Forest) e compare o ganho de AUC-PR contra o custo de tempo de treino.

---